# GeoDiff-TrustMoE residual recovery on your Landsat/Sentinel tiles

This notebook accepts either:

1. Raw Landsat 8/9 Collection-2 Level-2 products and Sentinel-2 L2A SAFE
   products or archives anywhere under `/kaggle/input`; or
2. The portable prepared dataset created by
   `Landsat_Sentinel_TrustMoE_Dataset_Preparation.ipynb`.

Raw products are paired by geographic overlap and acquisition date. The
resulting examples contain actual Landsat 30 m RGB inputs and actual
Sentinel-2 10 m RGB targets. Stored geometry is 128x128 -> 384x384. Fixed
within-tile spatial regions provide train, validation and at least 10% test
pairs. Existing preparation state resumes and newly attached products are
included. Invalid pairs are written to a quarantine log.

The study first tests whether one residual expert can learn, then compares
dense, uniform, base-error-routed, and correction-benefit-routed models. The
target HR image supplies training labels only and never enters inference.
Every model uses the same frozen base and fixed splits. There is one base
pass, no diffusion, VAE, PixelShuffle, or synthetic LR generation.

Keep test evaluation disabled until validation is reviewed. Results include
per-category and per-tile reports, paired and tile-cluster intervals, routing,
parameter counts, measured runtime, visualizations, exact predictions, and a
restart/download ZIP. Run cells from top to bottom.


## 1. Controls: paths, epochs, batch size, experts, and scene labels


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, tarfile, time, zipfile

FAST_DEV_RUN = False
REPOSITORY_URL = 'https://github.com/shashankjs2002/SI-SR-1.git'
REPOSITORY_BRANCH = '3x-continued'
REPOSITORY_DIR = Path('/kaggle/working/geodiff-trust-recovery-tiles-source')
SUITE_ROOT = Path('/kaggle/working/geodiff-trust-recovery-tiles-v2')
RAW_TILE_ROOT = Path('/kaggle/input')
PREPARED_MANIFEST = None  # Optional exact /kaggle/input/.../manifest.jsonl.
RESTORE_SUITE_FROM = None  # Prior Kaggle output folder or downloaded ZIP.
OLD_SUITE_ROOT = Path('/kaggle/working/geodiff-trust-moe-v1')
BASE_CHECKPOINTS = {42: None}  # Optional prior best.pt path per seed.

EPOCHS = {'base': 30, 'diagnostic': 60, 'single': 15, 'residual': 10}
BATCH_SIZE = 4
NUM_EXPERTS, TOP_K = 5, 2
REGION_FRACTION = 0.5
SEEDS = [42]  # Final study: [42, 123, 2026].
RUN = dict(dense_gain=True, sparse_error=True, sparse_gain=True,
           sparse_no_trust=True, sparse_uniform=True, sparse_conv=False,
           sparse_adversarial=False, adaptive_k=False,
           legacy_expert_control=False)
RUN_TEST_EVALUATION = False
REQUIRE_RECOVERY_SCREEN = True
DIAGNOSTIC_PAIRS = 8

# Used only when a class cannot be inferred from directories such as urban,
# agriculture, forest, water, barren, grassland, industrial, or mixed.
CATEGORY_BY_TILE = {
    # '44QLL': 'urban',
}
CATEGORY_BY_PRODUCT = {
    # 'S2A_MSIL2A_202...': 'agriculture',
}

# Fixed data protocol. Change only for a new prepared-dataset version.
PATCH_SIZE, PATCH_STRIDE = 384, 288
MAX_DAY_GAP = 3
MINIMUM_OVERLAP_FRACTION = 0.10
MINIMUM_VALID_FRACTION = 0.95
TRAIN_FRACTION, VALIDATION_FRACTION = 0.78, 0.10
MINIMUM_VALIDATION_FRACTION, MINIMUM_TEST_FRACTION = 0.08, 0.10
TRAIN_LR_CROP = 64
DISPLAY_MAX = 0.3
DATASET_PROTOCOL = 'own_tiles_spatial_v1'
BASE_MODEL = dict(base_embed_dim=32, base_depth=2, base_groups=2,
                  base_heads=4, window_size=8, width=32, tile_size=8)
MAX_PAIRS = 1 if FAST_DEV_RUN else None
REBUILD_PREPARED_DATA = False

if sys.version_info < (3, 10):
    raise RuntimeError('Use a Python 3.10+ Kaggle kernel.')
if not 1 <= TOP_K <= NUM_EXPERTS or not 0 < REGION_FRACTION <= 1:
    raise ValueError('Invalid expert count/top-k/region fraction')
if TRAIN_FRACTION + VALIDATION_FRACTION >= 1:
    raise ValueError('Train and validation fractions must leave test data')
SUITE_ROOT.mkdir(parents=True, exist_ok=True)

def safe_extract(path, root):
    root = Path(root).resolve(); root.mkdir(parents=True, exist_ok=True)
    def checked(name):
        target = (root / name).resolve()
        if not target.is_relative_to(root):
            raise ValueError('Unsafe archive member: ' + name)
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            for member in archive.infolist():
                checked(member.filename)
                if (member.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError('Archive symlinks are not accepted')
            archive.extractall(root)
    else:
        with tarfile.open(path) as archive:
            for member in archive.getmembers():
                checked(member.name)
                if not (member.isfile() or member.isdir()):
                    raise ValueError('Only regular files/directories are accepted')
            archive.extractall(root, filter='data')

if RESTORE_SUITE_FROM:
    source = Path(RESTORE_SUITE_FROM)
    if source.is_file():
        expanded = SUITE_ROOT.parent / 'trust-recovery-tiles-restore'
        safe_extract(source, expanded); source = expanded
    if not source.is_dir():
        raise FileNotFoundError(source)
    for path in source.rglob('*'):
        if path.is_file():
            destination = SUITE_ROOT / path.relative_to(source)
            if not destination.exists():
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, destination)
    print('Restored missing files without replacing current files.')

def run(command, cwd=None):
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(REPOSITORY_DIR / 'src') + os.pathsep + environment.get('PYTHONPATH', '')
    environment['PYTHONUNBUFFERED'] = '1'
    print('+', ' '.join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, env=environment, check=True)

print('Suite:', SUITE_ROOT)
print('Epochs:', EPOCHS, 'batch:', BATCH_SIZE, 'experts:', NUM_EXPERTS, 'top-k:', TOP_K)


## 2. Clone the branch, install it, and verify the GPU


In [ ]:
if not REPOSITORY_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch',
         REPOSITORY_BRANCH, REPOSITORY_URL, REPOSITORY_DIR])
if not (REPOSITORY_DIR / '.git').is_dir():
    raise RuntimeError('REPOSITORY_DIR is not the requested Git checkout')
branch = subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'branch', '--show-current'], text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f'Expected {REPOSITORY_BRANCH}, found {branch}; use another path.')
required = REPOSITORY_DIR / 'src/geodiff_gan/experiments/trust_recovery.py'
if not required.is_file():
    raise RuntimeError('Push the current 3x-continued changes before running this notebook.')
run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'rasterio>=1.3',
     'Pillow', 'PyYAML', 'tqdm', 'pandas', 'matplotlib'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPOSITORY_DIR, '--no-deps'])
sys.path.insert(0, str(REPOSITORY_DIR / 'src'))

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, FileLink
from geodiff_gan.data.diverse import (
    annotate_scene_classes, infer_scene_class, normalize_scene_class,
    validate_category_splits, write_dataset_card,
)
from geodiff_gan.data.manifest import load_manifest, write_manifest
from geodiff_gan.data.sentinel import discover_safe_products, tile_id_from_product
from geodiff_gan.data.landsat_sentinel import discover_landsat_products, pair_scenes
from geodiff_gan.experiments.trust_moe import (
    make_config, prepare_manifest, write_json, digest_file, adopt_base,
    evaluate, dataset_for, load_model,
)
from geodiff_gan.experiments.trust_recovery import (
    RECOVERY_VERSION, recovery_config, audit_checkpoint,
    memorization_report, validation_screen,
)
from geodiff_gan.experiments.trust_report import (
    METRICS, summarize, benchmark, compare_controls, paired_intervals,
    visualize, bundle_results,
)
if RECOVERY_VERSION != 'trust-recovery-v2':
    raise RuntimeError('Wrong source revision')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator.')
for name in ('figures', 'reports', 'diagnostics', 'source_receipts'):
    (SUITE_ROOT / name).mkdir(exist_ok=True)
commit = subprocess.check_output(['git', '-C', str(REPOSITORY_DIR), 'rev-parse', 'HEAD'], text=True).strip()
write_json(SUITE_ROOT / 'source_receipts/revision.json', dict(commit=commit, branch=branch))
print('Commit:', commit, 'Torch:', torch.__version__, 'GPU:', torch.cuda.get_device_name(0))
if FAST_DEV_RUN:
    EPOCHS = {'base': 1, 'diagnostic': 2, 'single': 1, 'residual': 3}
    REQUIRE_RECOVERY_SCREEN = RUN_TEST_EVALUATION = False
    print('SMOKE RUN: pipeline verification only.')


## 3. Find a prepared manifest or prepare the raw tiles

If exactly one portable paired manifest is attached, it is used directly.
Otherwise the cell extracts raw product archives, discovers complete products,
pairs them, and creates restart-safe patches under the suite. Source archives
remain read-only. Added products are included on the next run. Set
`PREPARED_MANIFEST` when Kaggle input contains multiple datasets.


In [ ]:
if PREPARED_MANIFEST is None:
    cards = list(RAW_TILE_ROOT.rglob('dataset_card.json'))
    candidates = sorted({p.parent / 'manifest.jsonl' for p in cards
                         if (p.parent / 'manifest.jsonl').exists()})
    if len(candidates) == 1:
        PREPARED_MANIFEST = candidates[0]
        print('Using portable prepared dataset:', PREPARED_MANIFEST)
    elif len(candidates) > 1:
        raise ValueError(f'Set PREPARED_MANIFEST; found: {candidates[:12]}')

if PREPARED_MANIFEST is None:
    build = SUITE_ROOT / 'data'
    extract_root = build / 'extracted_products'
    patch_root = build / 'patches'
    raw_manifest = build / 'raw_manifest.jsonl'
    classified_manifest = build / 'classified_manifest.jsonl'
    quarantine = build / 'quarantine.jsonl'
    for directory in (extract_root, patch_root):
        directory.mkdir(parents=True, exist_ok=True)
    suffixes = ('.zip', '.tar', '.tar.gz', '.tgz')
    archives = [p for p in RAW_TILE_ROOT.rglob('*') if p.is_file()
                and p.name.casefold().endswith(suffixes)]
    print('Raw archives:', len(archives))
    for number, archive in enumerate(sorted(archives), 1):
        relative = archive.parent.relative_to(RAW_TILE_ROOT)
        identity = hashlib.sha256((str(archive.relative_to(RAW_TILE_ROOT)) +
                                   str(archive.stat().st_size)).encode()).hexdigest()[:12]
        destination = extract_root / relative / identity
        marker = destination / '.complete.json'
        if marker.exists():
            continue
        print(f'Extracting {number}/{len(archives)}: {archive.name}')
        safe_extract(archive, destination)
        marker.write_text(json.dumps({'archive': str(archive),
                                       'size': archive.stat().st_size}, indent=2))

    sentinels, landsats = {}, {}
    for root in (RAW_TILE_ROOT, extract_root):
        for product in discover_safe_products(root):
            sentinels.setdefault(product.name.casefold(), product)
        for product in discover_landsat_products(root):
            landsats.setdefault(product.product_id.casefold(), product)
    sentinels, landsats = list(sentinels.values()), list(landsats.values())
    if not sentinels or not landsats:
        raise RuntimeError('No complete Sentinel-2 L2A SAFE and Landsat C2 L2 products found')
    pairs, unmatched = pair_scenes(sentinels, landsats, max_day_gap=MAX_DAY_GAP,
                                   minimum_overlap_fraction=MINIMUM_OVERLAP_FRACTION)

    def category_for(product):
        tile = tile_id_from_product(product)
        if tile in CATEGORY_BY_TILE:
            return normalize_scene_class(CATEGORY_BY_TILE[tile])
        matches = [(prefix, value) for prefix, value in CATEGORY_BY_PRODUCT.items()
                   if product.name.casefold().startswith(prefix.casefold())]
        return (normalize_scene_class(max(matches, key=lambda x: len(x[0]))[1])
                if matches else infer_scene_class(product))

    pair_categories, pair_rows = {}, []
    for pair in pairs:
        category = category_for(pair.sentinel)
        pair_categories[pair.sentinel.name.casefold()] = category
        pair_rows.append(dict(scene_class=category or 'UNLABELED',
            tile_id=tile_id_from_product(pair.sentinel), sentinel=pair.sentinel.name,
            landsat=pair.landsat.product_id, day_gap=pair.day_gap,
            overlap=pair.overlap_fraction))
    pair_table = pd.DataFrame(pair_rows)
    display(pair_table)
    if pair_table.empty or pair_table.scene_class.eq('UNLABELED').any():
        display(pair_table[pair_table.scene_class.eq('UNLABELED')])
        raise ValueError('Label every source using class directories or CATEGORY overrides')
    pair_table.to_csv(build / 'source_pairs.csv', index=False)
    print('Products:', len(sentinels), 'Sentinel,', len(landsats), 'Landsat')
    print('Compatible pairs:', len(pairs), 'unmatched Sentinel:', len(unmatched))
    print('Pairs by class:', pair_table.scene_class.value_counts().to_dict())

    discovery = SUITE_ROOT / 'data/discovery'
    for sensor, products in (('sentinel', sentinels), ('landsat', landsats)):
        root = discovery / sensor; root.mkdir(parents=True, exist_ok=True)
        for product in products:
            identity = product.name if sensor == 'sentinel' else product.product_id
            destination = root / identity
            if sensor == 'sentinel':
                if not destination.exists():
                    destination.symlink_to(Path(product).resolve(), target_is_directory=True)
            else:
                destination.mkdir(parents=True, exist_ok=True)
                for source in product.files.values():
                    link = destination / Path(source).name
                    if not link.exists():
                        link.symlink_to(Path(source).resolve())
    command = [sys.executable, '-m', 'geodiff_gan.cli.prepare_landsat_sentinel',
        '--sentinel-input', discovery / 'sentinel', '--landsat-input', discovery / 'landsat',
        '--output', patch_root, '--manifest', raw_manifest,
        '--state', build / 'pairing_state.json', '--quarantine', quarantine,
        '--patch-size', PATCH_SIZE, '--stride', PATCH_STRIDE,
        '--max-day-gap', MAX_DAY_GAP,
        '--minimum-overlap-fraction', MINIMUM_OVERLAP_FRACTION,
        '--minimum-valid-fraction', MINIMUM_VALID_FRACTION,
        '--bandpass-adjustment', 'none', '--split-strategy', 'within-tile-spatial',
        '--train-fraction', TRAIN_FRACTION, '--validation-fraction', VALIDATION_FRACTION,
        '--minimum-validation-fraction', MINIMUM_VALIDATION_FRACTION,
        '--minimum-test-fraction', MINIMUM_TEST_FRACTION]
    if MAX_PAIRS is not None:
        command += ['--max-pairs', MAX_PAIRS]
    if REBUILD_PREPARED_DATA:
        command.append('--rebuild')
    run(command, REPOSITORY_DIR)
    records = annotate_scene_classes(load_manifest(raw_manifest),
        category_by_tile=CATEGORY_BY_TILE,
        category_by_product={**CATEGORY_BY_PRODUCT,
            **{k: v for k, v in pair_categories.items() if v}}, require_all=True)
    write_manifest(classified_manifest, records)
    category_rows = validate_category_splits(records,
        minimum_test_fraction=MINIMUM_TEST_FRACTION, require_each_split=True)
    pd.DataFrame(category_rows).to_csv(build / 'category_split_summary.csv', index=False)
    write_dataset_card(build, records, preparation=dict(patch_size=PATCH_SIZE,
        stride=PATCH_STRIDE, train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION, minimum_test_fraction=MINIMUM_TEST_FRACTION,
        max_day_gap=MAX_DAY_GAP, minimum_valid_fraction=MINIMUM_VALID_FRACTION,
        input='actual Landsat 30 m RGB', target='actual Sentinel-2 10 m RGB'))
    PREPARED_MANIFEST = classified_manifest

PREPARED_MANIFEST = Path(PREPARED_MANIFEST)
MANIFEST = SUITE_ROOT / 'runtime_manifest.jsonl'
AUDIT = prepare_manifest(PREPARED_MANIFEST, MANIFEST, spatial_audit=True,
                         minimum_test_fraction=MINIMUM_TEST_FRACTION)
DATA_CARD = {'protocol': DATASET_PROTOCOL, 'source_manifest': str(PREPARED_MANIFEST)}
AUDIT['input'] = 'actual Landsat 8/9 Collection-2 L2 RGB at 30 m'
AUDIT['target'] = 'actual Sentinel-2 L2A RGB at 10 m'
write_json(SUITE_ROOT / 'dataset_audit.json', AUDIT)
print(json.dumps(AUDIT, indent=2))
if AUDIT['frames_hr'] != [PATCH_SIZE]:
    raise RuntimeError(f'Expected stored HR {PATCH_SIZE}, found {AUDIT["frames_hr"]}')
records = load_manifest(MANIFEST)
table = pd.DataFrame([dict(split=r.split, tile=r.tile_id,
                           category=r.scene_class) for r in records])
display(pd.crosstab([table.category, table.tile], table.split))
print('Test fraction:', AUDIT['counts']['test'] / sum(AUDIT['counts'].values()))
print('Do not inspect test predictions before the frozen-plan section.')


## 4. Visual check using training data only


In [ ]:
from torch.nn import functional as F

def preview_pair(index=0):
    config = make_config(MANIFEST, SUITE_ROOT, profile='base', crop_size=TRAIN_LR_CROP)
    data = dataset_for(config, 'train')
    sample = data[int(index) % len(data)]
    lr, hr = sample['lr'], sample['hr']
    bicubic = F.interpolate(lr[None], size=hr.shape[-2:], mode='bicubic',
                            align_corners=False)[0].clamp(0, 1)
    fig, axes = plt.subplots(1, 4, figsize=(17, 4))
    values = ((lr, 'Original Landsat 30 m'), (bicubic, 'Bicubic display'),
              (hr, 'Sentinel-2 target 10 m'), (sample['valid_mask'], 'Valid mask'))
    for axis, (array, title) in zip(axes, values):
        if array.shape[0] == 1:
            axis.imshow(array[0], vmin=0, vmax=1, cmap='gray')
        else:
            axis.imshow(np.clip(array.permute(1, 2, 0) / DISPLAY_MAX, 0, 1) ** (1 / 1.4),
                        interpolation='nearest')
        axis.set_title(f'{title}\n{array.shape[-2]} x {array.shape[-1]}')
        axis.axis('off')
    fig.tight_layout()
    fig.savefig(SUITE_ROOT / f'figures/train_pair_{index}.png', dpi=170)
    plt.show()
    record = data.records[int(index) % len(data)]
    print('Class:', record.scene_class, 'tile:', record.tile_id,
          'source:', record.source_product)
    print('Stored range LR:', float(lr.min()), float(lr.max()),
          'HR:', float(hr.min()), float(hr.max()))
preview_pair(0)


## 5. Reuse your trained base and audit the old residual

The imported base is copied into this suite. All comparisons use exactly those
frozen weights. If no base can be found, this cell trains one. Attach your prior
Kaggle outputs and set OLD_SUITE_ROOT to avoid repeating base training.
Epoch increases resume; architecture/loss changes need a new experiment directory.


In [ ]:
BASES, SINGLES, MODELS, CONFIGS = {}, {}, {}, {}

def launch(config):
    root = Path(config['root']); root.mkdir(parents=True, exist_ok=True)
    request = root / 'request_config.json'
    write_json(request, config)
    run([sys.executable, '-m', 'geodiff_gan.cli.trust_moe', 'train', '--config', request], cwd=REPOSITORY_DIR)
    checkpoint = root / 'best.pt'
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    return checkpoint

def read_checkpoint_config(checkpoint):
    return torch.load(checkpoint, map_location='cpu', weights_only=False)['config']

for seed in SEEDS:
    config = make_config(MANIFEST, SUITE_ROOT / f'seed_{seed}/base', profile='base',
        seed=seed, epochs=EPOCHS['base'], batch_size=BATCH_SIZE, crop_size=TRAIN_LR_CROP,
        experts=NUM_EXPERTS, top_k=TOP_K, dataset_id=AUDIT['dataset_id'], base_model=BASE_MODEL)
    config['training']['progress'] = 'compact'
    requested = BASE_CHECKPOINTS.get(seed)
    old = Path(requested) if requested else OLD_SUITE_ROOT / f'seed_{seed}/base/best.pt'
    if old.is_file():
        prior = read_checkpoint_config(old)
        if prior['dataset_id'] != AUDIT['dataset_id']:
            raise ValueError('Old base dataset differs. Check data protocol and manifest before importing.')
        # Honor the real base dimensions recorded by the trained checkpoint.
        for key in ('base_embed_dim', 'base_depth', 'base_groups', 'base_heads', 'window_size'):
            if key in prior['model']:
                config['model'][key] = prior['model'][key]
        BASES[seed] = adopt_base(config, old)
        print('Imported trained base:', old)
    else:
        print('No imported base found; training the base for this seed.')
        BASES[seed] = launch(config)
    MODELS[f'{seed}/base'] = BASES[seed]
    CONFIGS[f'{seed}/base'] = config
    write_json(SUITE_ROOT / f'seed_{seed}/base/lineage.json', {
        'source': str(old) if old.is_file() else 'trained in this suite',
        'source_sha256': digest_file(old) if old.is_file() else None,
        'checkpoint_sha256': digest_file(BASES[seed]),
    })
    old_expert = OLD_SUITE_ROOT / f'seed_{seed}/single_expert/best.pt'
    if old_expert.exists():
        previous = audit_checkpoint(old_expert, MANIFEST, SUITE_ROOT / f'diagnostics/old_single_{seed}.json')
        display(pd.DataFrame(previous['rows']))
        print('Reconstruction gradients:', previous.get('reconstruction_gradients'))
        print('Full objective gradients:', previous.get('full_loss_gradients'))
write_json(SUITE_ROOT / 'base_registry.json', {str(k): str(v.relative_to(SUITE_ROOT)) for k, v in BASES.items()})


## 6. Eight-image training diagnostic: can an expert learn a correction?

This uses fixed TRAIN crops, full coverage, one expert, trust off and guard off.
Its PSNR is memorization performance, never a validation/test score. It is not
used to initialize the scientific runs. A failed screen stops the full study so
you can share the diagnostic ZIP. No target is passed into the inference model.


In [ ]:
def new_config(seed, label, profile='single_expert', initializer=None, risk='gain', hr=True):
    dimensions = {k: v for k, v in CONFIGS[f'{seed}/base']['model'].items()
                  if k.startswith('base_') or k == 'window_size'}
    dimensions.update(width=BASE_MODEL['width'], tile_size=BASE_MODEL['tile_size'])
    return recovery_config(MANIFEST, SUITE_ROOT / f'seed_{seed}/{label}', parent=BASES[seed],
        initializer=initializer, profile=profile, seed=seed,
        epochs=EPOCHS['single'] if profile == 'single_expert' else EPOCHS['residual'],
        batch_size=BATCH_SIZE, crop_size=TRAIN_LR_CROP, experts=NUM_EXPERTS,
        top_k=TOP_K, dataset_id=AUDIT['dataset_id'], base_model=dimensions,
        risk_target=risk, coverage=REGION_FRACTION, hr_conditioning=hr)

DIAGNOSTICS = []
for seed in SEEDS:
    diagnostic = new_config(seed, 'diagnostic_overfit')
    diagnostic['training'].update(diagnostic_overfit_pairs=DIAGNOSTIC_PAIRS,
        epochs=EPOCHS['diagnostic'], batch_size=1, num_workers=0, learning_rate=1e-3,
        lr_epoch_decay=1.0)
    path = launch(diagnostic)
    report = memorization_report(path, SUITE_ROOT / f'diagnostics/memorization_{seed}.json')
    DIAGNOSTICS.append(report)
    display(pd.DataFrame(report['rows']))
    print('TRAIN-CROP diagnostic:', report['mean_psnr_gain'], 'dB; passed:', report['passed'])
if REQUIRE_RECOVERY_SCREEN and not all(r['passed'] for r in DIAGNOSTICS):
    archive = SUITE_ROOT.parent / (SUITE_ROOT.name + '_diagnostic.zip')
    bundle_results(SUITE_ROOT, archive, source_root=REPOSITORY_DIR)
    display(FileLink(str(archive.relative_to(Path('/kaggle/working')))))
    raise RuntimeError('Expert did not pass the train-crop learning check. Share diagnostics before running the full suite.')


## 7. Full training: one HR-conditioned residual expert

This trains on the complete fixed training split, using independent fresh
residual initialization and the common frozen base. Its checkpoint is selected
on the complete fixed validation split. This is the initializer for every subsequent MoE.


In [ ]:
for seed in SEEDS:
    config = new_config(seed, 'single_recovery')
    CONFIGS[f'{seed}/single_recovery'] = config
    SINGLES[seed] = launch(config)
    MODELS[f'{seed}/single_recovery'] = SINGLES[seed]


## 8. Validate the single expert before training all routers

Development screen: mean validation gain at least +0.01 dB, and over half the
images improve. This detects useful learning; it is not a publication criterion.
The report compares raw/EMA behavior and logs individual-module gradient norms.


In [ ]:
VAL_EVALS, SINGLE_SCREENS = {}, []

def eval_path(name, split, checkpoint, suffix='trained'):
    return SUITE_ROOT / 'evaluation' / split / name / (digest_file(checkpoint)[:12] + '_' + suffix)

for seed in SEEDS:
    for label, checkpoint in (('base', BASES[seed]), ('single_recovery', SINGLES[seed])):
        name = f'{seed}/{label}'
        output = eval_path(name, 'val', checkpoint)
        evaluate(checkpoint, output, 'val', manifest=MANIFEST)
        VAL_EVALS[name] = output
    rows = json.loads((VAL_EVALS[f'{seed}/single_recovery'] / 'per_image.json').read_text())
    screen = validation_screen(rows)
    SINGLE_SCREENS.append(screen)
    audit = audit_checkpoint(SINGLES[seed], MANIFEST, SUITE_ROOT / f'diagnostics/single_recovery_{seed}.json')
    display(pd.DataFrame(audit['rows']))
    display(pd.DataFrame(paired_intervals(rows)))
    print('Validation development screen:', screen)
write_json(SUITE_ROOT / 'reports/single_screen.json', SINGLE_SCREENS)
if REQUIRE_RECOVERY_SCREEN and not all(s['passed'] for s in SINGLE_SCREENS):
    raise RuntimeError('Single expert did not improve validation enough. Run the final bundle cell and share the evidence.')


## 9. Shared initializer and matched routing experiments

Each MoE starts from the same learned encoder/single expert. Tiny independent RGB
projection perturbations break symmetry. Two full-coverage warm-up epochs observe
proposal benefit throughout the image; later epochs use the declared sparse budget.
The router's auxiliary loss is prevented from updating the reconstruction encoder.
Warm-up checkpoints cannot become the selected deployed checkpoint.

`sparse_error` predicts base error; `sparse_gain` predicts realized proposal benefit.
Both use identical architectures, data, losses, training duration, and inference
budget except the risk label. Unexecuted tiles are excluded from gain supervision.
This label remains an imperfect, changing estimate of recoverability, not an oracle.


In [ ]:
EXPERIMENTS = {
    'dense_gain': ('dense_transformer', 'gain'),
    'sparse_error': ('sparse_transformer', 'error'),
    'sparse_gain': ('sparse_transformer', 'gain'),
    'sparse_no_trust': ('sparse_no_trust', 'gain'),
    'sparse_uniform': ('sparse_uniform', 'gain'),
    'sparse_conv': ('sparse_conv', 'gain'),
    'sparse_adversarial': ('sparse_adversarial', 'gain'),
    'adaptive_k': ('adaptive_k', 'gain'),
}

def train_experiment(label):
    if not RUN.get(label, False):
        print('Skipped:', label)
        return
    profile, risk = EXPERIMENTS[label]
    for seed in SEEDS:
        config = new_config(seed, label, profile, initializer=SINGLES[seed], risk=risk)
        CONFIGS[f'{seed}/{label}'] = config
        MODELS[f'{seed}/{label}'] = launch(config)
    write_json(SUITE_ROOT / 'model_registry.json', {k: str(v.relative_to(SUITE_ROOT)) for k, v in MODELS.items()})


## 10. Dense MoE control


In [ ]:
train_experiment('dense_gain')


## 11. Sparse base-error router control


In [ ]:
train_experiment('sparse_error')


## 12. Sparse proposal-benefit router


In [ ]:
train_experiment('sparse_gain')


## 13. Remove trust


In [ ]:
train_experiment('sparse_no_trust')


## 14. Uniform regional coverage control


In [ ]:
train_experiment('sparse_uniform')


## 15. CNN router control (optional)


In [ ]:
train_experiment('sparse_conv')


## 16. Adversarial contribution (optional)


In [ ]:
train_experiment('sparse_adversarial')


## 17. Adaptive expert slots (optional)


In [ ]:
train_experiment('adaptive_k')


## 18. Legacy expert with the new curriculum (optional)

This control isolates the HR-conditioned expert architecture. It receives the
same single-expert loss and duration but has the old expert capacity and no HR
cues. It is trained from scratch using the shared base, so it does not inherit
an incompatible HR expert checkpoint. Report parameter differences explicitly.


In [ ]:
if RUN['legacy_expert_control']:
    for seed in SEEDS:
        config = new_config(seed, 'legacy_expert_control', hr=False)
        CONFIGS[f'{seed}/legacy_expert_control'] = config
        MODELS[f'{seed}/legacy_expert_control'] = launch(config)


## 19. Validation comparisons, routing, learning curves and measured runtime


In [ ]:
TIMINGS, HISTORIES = [], []
for name, checkpoint in MODELS.items():
    output = eval_path(name, 'val', checkpoint)
    evaluate(checkpoint, output, 'val', manifest=MANIFEST)
    VAL_EVALS[name] = output
    for row in benchmark(checkpoint, output / 'timing.json', manifest=MANIFEST):
        TIMINGS.append(dict(experiment=name, **row))
    for history in sorted(checkpoint.parent.glob('history/epoch_*.json')):
        row = json.loads(history.read_text())
        diagnostics = row.pop('first_batch_diagnostics', {})
        row.pop('weighted_objective', None)
        row.pop('first_batch_gradient_norms', None)
        HISTORIES.append(dict(experiment=name, **row, **diagnostics))
VAL_TABLES = summarize(VAL_EVALS, SUITE_ROOT / 'reports/validation')
display(VAL_TABLES['overall'][['experiment', 'count', *METRICS, 'psnr_delta_vs_base', 'correction_abs_mean']])
display(VAL_TABLES['expert_usage'])
TIMING_TABLE = pd.DataFrame(TIMINGS)
TIMING_TABLE.to_csv(SUITE_ROOT / 'reports/timing.csv', index=False)
display(TIMING_TABLE[['experiment', 'operation', 'parameters_total', 'parameters_used_on_this_frame',
                     'mean_ms', 'p95_ms', 'images_per_second', 'peak_allocated_mb']])
history = pd.DataFrame(HISTORIES)
history.to_csv(SUITE_ROOT / 'reports/training_history.csv', index=False)
if not history.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for name, group in history.groupby('experiment'):
        axes[0].plot(group.epoch, group.val_psnr, label=name)
        if 'correction_abs_mean' in group:
            axes[1].plot(group.epoch, group.correction_abs_mean, label=name)
    axes[0].set_title('Validation PSNR'); axes[1].set_title('First training batch: effective correction')
    axes[0].legend(fontsize=6); fig.tight_layout()
    fig.savefig(SUITE_ROOT / 'figures/learning.png', dpi=170); plt.show()
for seed in SEEDS:
    group = {k: v for k, v in VAL_EVALS.items() if k.startswith(f'{seed}/')}
    if f'{seed}/sparse_gain' in group:
        display(compare_controls(group, f'{seed}/sparse_gain', SUITE_ROOT / f'reports/validation/controls_{seed}.csv'))
write_json(SUITE_ROOT / 'model_registry.json', {k: str(v.relative_to(SUITE_ROOT)) for k, v in MODELS.items()})


## 20. Validation-only coverage ablation

Trained top-k stays fixed. Evaluate 25/50/75/100% coverage to measure a quality vs
computation curve. Final primary comparisons retain the trained coverage; these
curves are diagnostic ablations. A reduced fraction is not proof of lower latency.


In [ ]:
BUDGET_ROWS = []
for seed in SEEDS:
    name = f'{seed}/sparse_gain'
    if name not in MODELS:
        continue
    for fraction in (0.25, 0.5, 0.75, 1.0):
        output = eval_path(name, 'val', MODELS[name], f'coverage_{fraction}')
        scores = evaluate(MODELS[name], output, 'val', coverage=fraction,
                          manifest=MANIFEST, save_images=False)
        timing = benchmark(MODELS[name], output / 'timing.json', manifest=MANIFEST, coverage=fraction)[1]
        BUDGET_ROWS.append(dict(seed=seed, coverage=fraction, **scores,
                               mean_ms=timing['mean_ms'], images_per_second=timing['images_per_second']))
budget = pd.DataFrame(BUDGET_ROWS)
budget.to_csv(SUITE_ROOT / 'reports/validation_coverage.csv', index=False)
display(budget)


## 21. Freeze the test plan after reviewing validation

The proposed primary model is sparse_gain at the trained coverage and top-k.
Record whether it improves over both the base and the strong single expert;
failure is retained in the report. All enabled controls are tested. No checkpoint
or budget is chosen by looking at test scores. Keep this tile test split locked.
A geographically independent test region is still needed for a strong final claim when current splits share source tiles.


In [ ]:
plan_path = SUITE_ROOT / 'test_plan.json'
if plan_path.exists():
    TEST_PLAN = json.loads(plan_path.read_text())
    if TEST_PLAN['dataset_id'] != AUDIT['dataset_id']:
        raise ValueError('Test plan belongs to different data')
else:
    TEST_PLAN = dict(dataset_id=AUDIT['dataset_id'], protocol=DATASET_PROTOCOL,
                     primary='sparse_gain', coverage=REGION_FRACTION, top_k=TOP_K,
                     development_screens=SINGLE_SCREENS, checkpoints={})
    for name, source in MODELS.items():
        identity = digest_file(source)
        destination = SUITE_ROOT / 'test_models' / name / identity[:12] / 'best.pt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy2(source, destination)
        if digest_file(destination) != identity:
            raise RuntimeError('Test checkpoint copy differs')
        TEST_PLAN['checkpoints'][name] = dict(path=str(destination.relative_to(SUITE_ROOT)), sha256=identity)
    write_json(plan_path, TEST_PLAN)
print(json.dumps(TEST_PLAN, indent=2))


## 22. Test the complete fixed test split using the frozen plan


In [ ]:
TEST_EVALS = {}
if RUN_TEST_EVALUATION:
    if FAST_DEV_RUN:
        raise RuntimeError('Smoke results must not be reported as the official test benchmark')
    for name, item in TEST_PLAN['checkpoints'].items():
        checkpoint = SUITE_ROOT / item['path']
        if digest_file(checkpoint) != item['sha256']:
            raise RuntimeError('Test snapshot changed: ' + name)
        output = SUITE_ROOT / 'evaluation/test' / name
        scores = evaluate(checkpoint, output, 'test', manifest=MANIFEST)
        if scores['count'] != AUDIT['counts']['test']:
            raise RuntimeError('Expected the complete fixed test split')
        TEST_EVALS[name] = output
    TEST_TABLES = summarize(TEST_EVALS, SUITE_ROOT / 'reports/test')
    display(TEST_TABLES['overall'][['experiment', 'count', *METRICS, 'psnr_delta_vs_base', 'correction_abs_mean']])
    display(TEST_TABLES['paired_intervals'])
    for seed in SEEDS:
        group = {k: v for k, v in TEST_EVALS.items() if k.startswith(f'{seed}/')}
        if f'{seed}/sparse_gain' in group:
            display(compare_controls(group, f'{seed}/sparse_gain', SUITE_ROOT / f'reports/test/controls_{seed}.csv'))
else:
    print('Test is disabled. After validation, set RUN_TEST_EVALUATION=True and rerun this cell.')


## 23. Indexed outputs, signed corrections and routing maps

`show_result(index=25)` defaults to test. Use split='val' before opening test.
Images use the protocol range directly (gamma=1), with one shared scale and a
maximum of three columns. Correction plots use a shared symmetric color range;
they are signed correction amplitudes, not an artificially enhanced prediction.


In [ ]:
from geodiff_gan.experiments.trust_report import visualize_saved

def show_result(index=0, split='test', show_base=True, profiles=None, seed=42):
    profiles = profiles or ['single_recovery', 'sparse_error', 'sparse_gain']
    registry = TEST_EVALS if split == 'test' else VAL_EVALS
    choices = {name.split('/', 1)[1]: path for name, path in registry.items()
               if name.startswith(f'{seed}/') and name.split('/', 1)[1] in profiles}
    if not choices:
        raise ValueError('Run the corresponding evaluation cell first.')
    destination = SUITE_ROOT / f'figures/{split}_{seed}_{index:06d}.png'
    figures = visualize_saved(choices, MANIFEST, index=index, split=split, show_base=show_base,
                               output_path=destination, display_max=1.0, gamma=1.0)
    corrections = []
    for name, path in choices.items():
        with np.load(path / 'images' / f'{index:06d}.npz', allow_pickle=False) as values:
            corrections.append((name, (values['prediction'] - values['base']).mean(0)))
    limit = max(1e-5, max(float(np.quantile(np.abs(a), .99)) for _, a in corrections))
    columns = min(3, len(corrections))
    fig, axes = plt.subplots(int(np.ceil(len(corrections)/columns)), columns,
                             figsize=(5*columns, 4*int(np.ceil(len(corrections)/columns))), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (name, array) in zip(axes.flat, corrections):
        picture = ax.imshow(array, cmap='RdBu_r', vmin=-limit, vmax=limit)
        ax.set_title(name + ': mean signed RGB correction')
    fig.colorbar(picture, ax=list(axes.flat), shrink=.7, label='Protocol intensity units')
    fig.savefig(destination.with_name(destination.stem + '_corrections.png'), dpi=170)
    plt.show()
    return figures

show_result(0, split='val')


## 24. Download the evidence and restart bundle

This cell can run even after a failed development screen. Download the ZIP and
save the executed notebook. Reports distinguish training diagnostics, validation,
test, failure, and measured runtime. Best/last checkpoints and code are included.
Full source tiles and prepared input caches are excluded. Six paired TIFF previews
are included for interpreting results without transferring the full dataset.


In [ ]:
import rasterio
from rasterio.transform import Affine

if 'BASES' in globals() and BASES:
    cfg = read_checkpoint_config(next(iter(BASES.values())))
    cfg['manifest'] = str(MANIFEST)
    split = 'test' if globals().get('TEST_EVALS') else 'val'
    data = dataset_for(cfg, split)
    folder = SUITE_ROOT / 'example_inputs'; folder.mkdir(exist_ok=True)
    for index in range(min(6, len(data))):
        sample = data[index]
        for key in ('lr', 'hr'):
            array = sample[key].numpy().astype('float32')
            # Prepared arrays do not retain the source crop transform; do not invent a CRS.
            path = folder / f'{split}_{index:06d}_{key}.tif'
            with rasterio.open(path, 'w', driver='GTiff', width=array.shape[2], height=array.shape[1],
                               count=3, dtype='float32', compress='deflate', transform=Affine.identity()) as tif:
                tif.write(array)
                tif.update_tags(description='Prepared RGB example; source crop georeferencing unavailable')
documentation = REPOSITORY_DIR / 'learning/trust_moe/07_residual_recovery_v2.md'
if documentation.exists():
    shutil.copy2(documentation, SUITE_ROOT / 'research_protocol.md')
notebook = REPOSITORY_DIR / 'kaggle/GeoDiff_TrustMoE_Tiles_Residual_Recovery_3x.ipynb'
if notebook.exists():
    shutil.copy2(notebook, SUITE_ROOT / notebook.name)
archive = Path('/kaggle/working') / f'trust_recovery_results_{time.strftime("%Y%m%d_%H%M%S")}.zip'
bundle_results(SUITE_ROOT, archive, source_root=REPOSITORY_DIR, include_resume=True)
os.chdir('/kaggle/working')
print(f'Results and resume bundle: {archive} ({archive.stat().st_size / 2**20:.1f} MiB)')
display(FileLink(archive.name, result_html_prefix='Download results: '))
